In [1]:
#Setup and Libraries
import pandas as pd
import numpy as np

print(" Libraries loaded successfully. Ready to process biological data.")

 Libraries loaded successfully. Ready to process biological data.


In [2]:
# Data Ingestion (Rice)
print(" Loading Oryza sativa (Rice) Transcriptomic Matrix...")

# 1. Define the exact path to where you saved the file.
file_path_rice = "C:\\Users\\kumar\\Desktop\\CrossKingdom_Biomarker_Project\\01_Data\\Raw_Data\\Rice\\GSE153030_counts.kallisto.PB.DS.txt"

# 2. Read the file into a Pandas DataFrame
# We use index_col=0 to tell Pandas that the first column contains the Gene Names, not just numbers.
rice_df = pd.read_csv(file_path_rice, sep='\t', index_col=0)

# 3. Print a success message
print(" Rice Data Loaded Successfully!")

 Loading Oryza sativa (Rice) Transcriptomic Matrix...
 Rice Data Loaded Successfully!


In [3]:
# Examining the Data Dimensions
print(" Analyzing the p >> n Problem (Curse of Dimensionality):")

# Get the number of rows and columns
num_genes = rice_df.shape[0]     # Rows (Genes / Features 'p')
num_samples = rice_df.shape[1]   # Columns (Biological Samples 'n')

print(f"Total Genes (p): {num_genes:,}")
print(f"Total Samples (n): {num_samples}")

if num_genes > num_samples:
    print("\n WARNING: Extreme p >> n imbalance detected. Traditional statistics will fail.")
    print(" We must use Machine Learning (Random Forest) for this data.")

 Analyzing the p >> n Problem (Curse of Dimensionality):
Total Genes (p): 13,343
Total Samples (n): 6

 We must use Machine Learning (Random Forest) for this data.


In [4]:
# Visualizing the Raw Expression Matrix
print(" Displaying the first 5 genes and 5 biological samples:\n")

# .head(5) shows the top 5 rows. .iloc[:, :5] shows the first 5 columns.
display(rice_df.iloc[:5, :5])

 Displaying the first 5 genes and 5 biological samples:



,M,N,O,P,Q
ID,,,,,
B1,519.00000,521.0,568.0,336.0,432.0
B10,117.00000,90.0,186.0,51.0,47.0
B100,25663.93892,23921.1,19158.6,31112.6,31980.8
B1000,38.00000,22.0,37.0,45.0,38.0
B10000,167.00000,154.0,156.0,118.0,118.0


In [5]:
# Transposing the Matrix
print(" Transposing the Data Matrix...")

# We need rows to be Samples, and columns to be Genes
rice_df_ml = rice_df.T 

# Create the Target Labels (0 = Normal, 1 = Heat Stress)
# Based on the NCBI GEO metadata, the first 3 are Control, the last 3 are Heat Stress
labels = [0, 0, 0, 1, 1, 1]
rice_df_ml['Heat_Stress_Target'] = labels

print(f"New Shape: {rice_df_ml.shape[0]} Samples (rows), {rice_df_ml.shape[1]-1} Genes (columns)")
display(rice_df_ml.head())

 Transposing the Data Matrix...
New Shape: 6 Samples (rows), 13343 Genes (columns)


ID,B1,B10,B100,B1000,B10000,B10001,B10002,B10003,B10004,B10005,...,transcript/56400,transcript/5650,transcript/5822,transcript/6618,transcript/7421,transcript/7454,transcript/8276,transcript/9539,transcript/9660,Heat_Stress_Target
M,519.0,117.0,25663.93892,38.0,167.0,192.0,221.0,233.000000,62.0,37.0,...,0.0,15.0,1.0,9.0,0.0,1010.0,17.0,9.0,39.0,0
N,521.0,90.0,23921.10000,22.0,154.0,216.0,171.0,231.000000,74.0,42.0,...,0.0,18.0,0.0,11.0,0.0,475.0,32.0,7.0,36.0,0
O,568.0,186.0,19158.60000,37.0,156.0,244.0,185.0,226.000000,111.0,55.0,...,0.0,8.0,1.0,9.0,0.0,661.0,28.0,17.0,16.0,0
P,336.0,51.0,31112.60000,45.0,118.0,102.0,116.0,178.000001,28.0,10.0,...,0.0,4.0,0.0,112.0,0.0,921.0,50.0,3.0,28.0,1
Q,432.0,47.0,31980.80000,38.0,118.0,162.0,157.0,128.000000,43.0,3.0,...,0.0,5.0,5.0,6.0,0.0,660.0,24.0,7.0,4.0,1


In [6]:
# Removing Low-Variance Genes (Noise)
from sklearn.feature_selection import VarianceThreshold

print(" Filtering out low-variance (noise) genes...")

# Separate features (X) and target (y)
X_rice_raw = rice_df_ml.drop('Heat_Stress_Target', axis=1)
y_rice = rice_df_ml['Heat_Stress_Target']

# Initialize the filter (removing genes with zero or near-zero variance)
# We use a threshold of 0.1 to remove genes that barely change
variance_filter = VarianceThreshold(threshold=0.1)

# Fit the filter to the data
X_rice_filtered_array = variance_filter.fit_transform(X_rice_raw)

# Get the names of the genes that survived the filter
surviving_genes = X_rice_raw.columns[variance_filter.get_support()]

# Create a new DataFrame with the surviving genes
X_rice_filtered = pd.DataFrame(X_rice_filtered_array, columns=surviving_genes, index=X_rice_raw.index)

genes_removed = X_rice_raw.shape[1] - X_rice_filtered.shape[1]
print(f" Filter Complete! Removed {genes_removed:,} noisy genes.")
print(f" Remaining active genes for ML: {X_rice_filtered.shape[1]:,}")

 Filtering out low-variance (noise) genes...
 Filter Complete! Removed 148 noisy genes.
 Remaining active genes for ML: 13,195


In [7]:
# Z-Score Normalization
from sklearn.preprocessing import StandardScaler

print(" Applying Z-Score Normalization (StandardScaler)...")

scaler = StandardScaler()

# Fit and transform the filtered data
X_rice_normalized_array = scaler.fit_transform(X_rice_filtered)

# Rebuild the final Pandas DataFrame ready for the Random Forest
X_rice_final = pd.DataFrame(X_rice_normalized_array, columns=X_rice_filtered.columns, index=X_rice_filtered.index)

print(" Data successfully normalized onto a common mathematical baseline (Mean=0, SD=1).")
display(X_rice_final.head())

 Applying Z-Score Normalization (StandardScaler)...
 Data successfully normalized onto a common mathematical baseline (Mean=0, SD=1).


ID,B1,B10,B100,B1000,B10000,B10001,B10002,B10003,B10004,B10005,...,transcript/55921,transcript/55990,transcript/56077,transcript/5650,transcript/5822,transcript/6618,transcript/7454,transcript/8276,transcript/9539,transcript/9660
M,0.734326,0.492847,-0.211754,-0.056284,1.266479,0.351766,1.595364,0.974319,0.158104,0.567044,...,-0.447214,0.698253,0.492917,1.077391,-0.46291,-0.426087,1.502562,-0.727370,0.345611,1.244177
N,0.755977,-0.073403,-0.611738,-1.857375,0.575672,0.851004,0.005300,0.921890,0.564656,0.820944,...,-0.447214,1.645733,1.062951,1.587734,-0.92582,-0.374440,-1.054821,0.421109,-0.031419,0.992546
O,1.264772,1.939931,-1.704740,-0.168852,0.681950,1.433449,0.450518,0.790815,1.818194,1.481084,...,2.236068,-1.278185,-0.602532,-0.113410,-0.46291,-0.426087,-0.165712,0.114848,1.853733,-0.684997
P,-1.246730,-0.891320,1.038723,0.731693,-1.337331,-1.520379,-1.743770,-0.467499,-0.993795,-0.804017,...,-0.447214,0.189553,-1.786297,-0.793867,-0.92582,2.233731,1.077128,1.799283,-0.785480,0.321529
Q,-0.207488,-0.975209,1.237976,-0.056284,-1.337331,-0.272283,-0.439918,-1.778242,-0.485605,-1.159477,...,-0.447214,-1.043827,-0.186162,-0.623753,1.38873,-0.503558,-0.170492,-0.191413,-0.031419,-1.691522


In [8]:
# Save the clean, normalized data for the ML Engine
import os

print(" Exporting clean data for the ML Pipeline...")

# Safety check: Ensure the output folder actually exists
os.makedirs("C:\\Users\\kumar\\Desktop\\CrossKingdom_Biomarker_Project\\01_Data\\Processed_Data", exist_ok=True)

# Define the exact path
output_path = "C:\\Users\\kumar\\Desktop\\CrossKingdom_Biomarker_Project\\01_Data\\Processed_Data\\Rice_Cleaned_Normalized.csv"

# Re-attach the target labels to the clean data
X_rice_final['Heat_Stress_Target'] = y_rice.values

# Export the CSV
X_rice_final.to_csv(output_path)

print(f"Success! Clean data saved to: {output_path}")

 Exporting clean data for the ML Pipeline...
Success! Clean data saved to: C:\Users\kumar\Desktop\CrossKingdom_Biomarker_Project\01_Data\Processed_Data\Rice_Cleaned_Normalized.csv
